# Detailed table-of-contents exploration

**Open Library is our main source of table-of-contents (TOC) data.** A TOC lists a book's chapters and other sections. Our saved dataset combines Amazon book records with information from matched Open Library editions. An edition is a particular published version of a book.

We explore the 9,034 rows already downloaded from BigQuery into `data/raw/books.parquet`. Each row has an Amazon book identifier, `parent_asin`, and its matched Open Library information. More than one Amazon record can refer to the same Open Library edition, so 9,034 rows does not necessarily mean 9,034 different editions.

The aim is to understand which TOC text and details are available before deciding how to prepare them for recommendations. Every number below refers to this saved file.

Run the cells in order using the project's `.venv` kernel. This notebook reads the local file; it does not query BigQuery or contact Open Library.

## 1. Load the local snapshot

Objective: Load the saved books and record the file checksum before exploration. From `.parquet` file.

In [1]:
import hashlib
import json
import re
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

from bookpath.local_data import read_books_from_parquet

project_root = Path.cwd()
while not (project_root / "src/bookpath").is_dir():
    if project_root == project_root.parent:
        raise FileNotFoundError("Open this notebook inside the Bookpath repository.")
    project_root = project_root.parent

raw_path = project_root / "data/raw/books.parquet"


def file_checksum(path):
    with path.open("rb") as raw_file:
        return hashlib.file_digest(raw_file, "sha256").hexdigest()


checksum_before = file_checksum(raw_path)
books = read_books_from_parquet(raw_path)
print(f"Books: {len(books):,}; checksum: {checksum_before}")

Books: 9,034; checksum: 4acf6979b15215c28f2a45a1488ace4d749e2573f0fa4a0110e2933dd0194dd8


## 2. Check the two saved copies of each book's TOC

Objective: Check whether the separate `raw_toc_json` column kept the **same TOC contents and order** as the saved Open Library edition record. Equal quantities alone would not be enough: two lists can have the same length but different chapter titles.

Both columns below belong to the `books` DataFrame loaded from `data/raw/books.parquet`.

| Column in `books` | What one cell stores | What we compare after `json.loads` |
| --- | --- | --- |
| `raw_toc_json` | A JSON string containing just that book's TOC | The entire decoded list |
| `openlibrary_raw_edition_json` | A JSON string containing the saved Open Library edition record, including its TOC | The list under the dictionary key `table_of_contents` |

`json.loads` converts JSON text into Python objects so we can access their contents. The first line below creates `raw_tocs` from `books["raw_toc_json"]`: it holds one decoded TOC list per book row. **Section 3 reuses this variable.**

**Why do this?** If the two lists agree for a book, we can inspect its separate `raw_toc_json` column without losing information from the saved edition record's `table_of_contents` list. This checks the saved copies, not whether the source TOC is complete or correctly matched to the book. It does not compare the `toc_entries` column yet.

The final table answers a separate question: **what kind of items does each book's TOC list contain?** It summarizes the saved `raw_toc_shape` column. `list[dict]` means a list of dictionaries; `list[string]` means a list of plain strings. Each book record contributes once to this table.

In [ ]:
# Each cell starts as a JSON string; decode it into a Python list of TOC items.
raw_tocs = books["raw_toc_json"].map(json.loads)

# Each full Open Library JSON string becomes a Python dictionary.
openlibrary_records = books["openlibrary_raw_edition_json"].map(json.loads)
# Select the original TOC list from each dictionary.
openlibrary_tocs = openlibrary_records.map(lambda record: record.get("table_of_contents"))

# Pair the two lists for the same book. Equality checks their contents and order.
# strict=True also checks that we compare the same number of books on both sides.
source_matches = pd.Series(
    [raw == original for raw, original in zip(raw_tocs, openlibrary_tocs, strict=True)],
    index=books.index,
)
print(f"Book rows with identical TOC contents and order in the two saved columns: {source_matches.sum():,} / {len(books):,}")
print("TOC-list format per book record (not the number of TOC items):")
display(books["raw_toc_shape"].value_counts().rename("book_records").to_frame())

**Result for this file:** All 9,034 book rows have identical TOC lists in the two saved locations, including item contents and order. We can therefore use `raw_toc_json` for the item-by-item inspection that follows.

The format table counts **book records**: 9,030 have lists of dictionaries, and 4 have lists of strings. It does not count the individual items inside those lists; section 3 does that.

Objective: Display one book's original TOC items and its parsed entries as two tables. Change `book_asin` to inspect another book by its Amazon identifier. The default is Never Split the Difference, as in docs/data.md. Original item positions and parsed sequence numbers can differ when an empty item was omitted.

`books.loc[books["parent_asin"] == book_asin]` keeps only matching rows. `.iloc[0]` then selects the first matching row as a Pandas `Series`. In this snapshot each `parent_asin` identifies one row. `selected_book["title"]` gets its book-title string; `selected_book["toc_entries"][0]["text"]` gets the first parsed TOC entry's text. `where()` would keep nonmatching rows and replace their values with missing values, so it is not the row filter we need here.

In this Parquet snapshot, `toc_entries` is a NumPy array of dictionaries. `list()` converts that array to a Python list while keeping the same dictionaries in order. Passing this list to `DataFrame` gives one column per dictionary key (`text`, `page`, etc.). Passing the array directly gives a single column containing whole dictionaries.

In [3]:
book_asin = "0062407805"
# Filter by the book ID, then select its row as a Pandas Series.
selected_book = books.loc[books["parent_asin"] == book_asin].iloc[0]
# Decode the raw TOC from that same row into a Python list.
selected_raw_toc = json.loads(selected_book["raw_toc_json"])  # COLUMN lookup: read this cell from the selected book row.
raw_view = pd.DataFrame({
    "raw_position": range(1, len(selected_raw_toc) + 1),
    "raw_item": selected_raw_toc,
})

# NumPy array of dictionaries -> Python list of dictionaries -> table of fields.
# list() changes the container for display; it does not clean any entry text.
parsed_entries = list(selected_book["toc_entries"])  # COLUMN lookup: read this cell from the selected book row.
parsed_view = pd.DataFrame(parsed_entries)

print(f"Selected parent_asin: {selected_book['parent_asin']}")
print(f"Amazon title: {selected_book['title']}")
print(f"Open Library title: {selected_book['ol_title']}")
with pd.option_context("display.max_colwidth", None):
    print("Original TOC: each raw_item cell contains an entire original item.")
    display(raw_view)
    print("Parsed TOC: each dictionary field has its own column.")
    display(parsed_view)

Selected parent_asin: 0062407805
Amazon title: Never Split the Difference: Negotiating As If Your Life Depended On It
Open Library title: Never Split the Difference
Original TOC: each raw_item cell contains an entire original item.


,raw_position,raw_item
0,1,"{'level': 0, 'label': 'Chapter 1', 'title': 'The New Rules', 'pagenum': '1', 'subtitle': 'How to Become the Smartest Person in Any Room', 'type': {'key': '/type/toc_item'}}"
1,2,"{'level': 0, 'label': 'Chapter 2', 'title': 'Be a Mirror', 'pagenum': '23', 'subtitle': 'How to Quickly Establish Rapport', 'type': {'key': '/type/toc_item'}}"
2,3,"{'level': 0, 'label': 'Chapter 3', 'title': 'Don't Feel Their Pain, Label It', 'pagenum': '49', 'subtitle': 'How to Create Trust with Tactical Empathy', 'type': {'key': '/type/toc_item'}}"
3,4,"{'level': 0, 'label': 'Chapter 4', 'title': 'Beware ""Yes""-Master ""No""', 'pagenum': '74', 'subtitle': 'How to Generate Momentum and Make It Safe to Reveal the Real Stakes', 'type': {'key': '/type/toc_item'}}"
4,5,"{'level': 0, 'label': 'Chapter 5', 'title': 'Trigger the Two Words That Immediately Transform Any Negotiation', 'pagenum': '96', 'subtitle': 'How to Gain the Permission to Persuade', 'type': {'key': '/type/toc_item'}}"
5,6,"{'level': 0, 'label': 'Chapter 6', 'title': 'Bend Their Reality', 'pagenum': '113', 'subtitle': 'How to Shape What Is Fair', 'type': {'key': '/type/toc_item'}}"
6,7,"{'level': 0, 'label': 'Chapter 7', 'title': 'Create the Illusion of Control', 'pagenum': '140', 'subtitle': 'How to Calibrate Questions to Transform Conflict into Collaboration', 'type': {'key': '/type/toc_item'}}"
7,8,"{'level': 0, 'label': 'Chapter 8', 'title': 'Guarantee Execution', 'pagenum': '162', 'subtitle': 'How to Spot the Liars and Ensure Follow-Through from Everyone Else', 'type': {'key': '/type/toc_item'}}"
8,9,"{'level': 0, 'label': 'Chapter 9', 'title': 'Bargain Hard', 'pagenum': '188', 'subtitle': 'How to Get Your Price', 'type': {'key': '/type/toc_item'}}"
9,10,"{'level': 0, 'label': 'Chapter 10', 'title': 'Find the Black Swan', 'pagenum': '213', 'subtitle': 'How to Create Breakthroughs by Revealing the Unknown Unknowns', 'type': {'key': '/type/toc_item'}}"


Parsed TOC: each dictionary field has its own column.


,level,page,sequence,source_key,text
0,0,1,1,title,The New Rules
1,0,23,2,title,Be a Mirror
2,0,49,3,title,"Don't Feel Their Pain, Label It"
3,0,74,4,title,"Beware ""Yes""-Master ""No"""
4,0,96,5,title,Trigger the Two Words That Immediately Transform Any Negotiation
5,0,113,6,title,Bend Their Reality
6,0,140,7,title,Create the Illusion of Control
7,0,162,8,title,Guarantee Execution
8,0,188,9,title,Bargain Hard
9,0,213,10,title,Find the Black Swan


## 3. Inspect the individual items inside each book's TOC

Objective: Look inside the TOC lists decoded from the `raw_toc_json` column and build one inspection record per TOC item—not one per book.

Section 2 already ran:

```python
raw_tocs = books["raw_toc_json"].map(json.loads)
```

That is the column lookup. The loop below uses its result, `raw_tocs`, rather than looking up `raw_toc_json` again.

| Name in the code | What it holds |
| --- | --- |
| `raw_tocs` | One decoded TOC list for each book row |
| `raw_toc` | The current book's list while the outer loop runs |
| `item` | One dictionary or string inside that list |
| `raw_rows` | A new Python list of inspection dictionaries, one per item |
| `raw_items` | The DataFrame created from `raw_rows` |

Each inspection dictionary keeps the book identifier, the item's position, and the complete item under the key `"raw_item"`. `candidate_text` holds text selected from the item's first nonblank `title`, otherwise `value`, otherwise `label`; a string item is used directly. `candidate_source_key` records the selected key (or `None` for a string).

**Why build this table?** A single book can contain many TOC items. Giving each item its own row lets us inspect its format, text, and missing details individually. The table stays in memory; it does not change the saved book data.

In [ ]:
def candidate_text_and_key(item):
    if isinstance(item, str):
        return item, None
    for key in ["title", "value", "label"]:
        value = item.get(key)
        if isinstance(value, str) and value.strip():
            return value, key
    return None, None


# raw_tocs comes from books["raw_toc_json"].map(json.loads) in section 2.
raw_rows = []
for parent_asin, raw_toc in zip(books["parent_asin"], raw_tocs, strict=True):
    for raw_position, item in enumerate(raw_toc, start=1):
        text, source_key = candidate_text_and_key(item)
        raw_rows.append({
            "parent_asin": parent_asin,
            "raw_position": raw_position,
            "raw_item": item,  # Create a dictionary key; store the current TOC item.
            "item_type": type(item).__name__,
            "candidate_text": text,
            "candidate_source_key": source_key,
        })

raw_items = pd.DataFrame(raw_rows)
print(f"Individual TOC items across all {len(books):,} book records: {len(raw_items):,}")
display(raw_items["item_type"].value_counts().rename("toc_items").to_frame())

**How this result relates to section 2**

| Format of a book's decoded `raw_toc_json` list | Book records (section 2) | Items inside those lists (section 3) |
| --- | ---: | ---: |
| `list[dict]`: each item is a dictionary | 9,030 | 141,384 dictionary items |
| `list[string]`: each item is a plain string | 4 | 129 string items |
| Total | 9,034 | 141,513 TOC items |

**141,384 is the number of individual dictionary items, not books or dictionary keys.** A chapter stored as `{"label": "Chapter 1", "title": "The New Rules"}` contributes one item, even though it has two keys. One book's list can contribute many such items.

`str` in the Python output means “string”; it is the same kind of item called `string` in the saved `list[string]` label. The two tables describe the same data at different levels: whole book lists versus the items inside them. They do not show different versions of the TOC.

Objective: Print one inspection dictionary from `raw_rows` for the book selected in section 2. The default book is *Never Split the Difference* (`parent_asin = "0062407805"`).

**Where did `"raw_item"` come from?** The previous cell created it here:

```python
raw_rows.append({
    # Other keys are omitted from this explanation.
    "raw_item": item,
})
```

`"raw_item"` on the left is a dictionary key we chose—a string, not a variable that needs an earlier definition. `item` on the right is the variable supplied by the inner loop. It contains one TOC item decoded from the book's `raw_toc_json` cell.

For the first chapter of this book, `item` includes `{"label": "Chapter 1", "title": "The New Rules", ...}`. The inspection dictionary stores that complete item under `"raw_item"`. The names `"raw_item"` and `"candidate_text"` are our inspection keys, not keys added to the original TOC.

The loop below calls one inspection dictionary `raw_row`. `pprint(raw_row)` prints all its keys and values, including the nested `"raw_item"` value. `break` stops after the first matching item, so it prints one item rather than the book's whole TOC.

In [5]:
from pprint import pprint

for raw_row in raw_rows:
    if raw_row["parent_asin"] == book_asin:
        pprint(raw_row, sort_dicts=False)
        break


{'parent_asin': '0062407805',
 'raw_position': 1,
 'raw_item': {'level': 0,
              'label': 'Chapter 1',
              'title': 'The New Rules',
              'pagenum': '1',
              'subtitle': 'How to Become the Smartest Person in Any Room',
              'type': {'key': '/type/toc_item'}},
 'item_type': 'dict',
 'candidate_text': 'The New Rules',
 'candidate_source_key': 'title'}


Objective: Find which dictionary keys occur inside TOC items decoded from the `books["raw_toc_json"]` column. Here, `property` means a dictionary key, such as `title` or `label`—not a separate column in `books`.

- `items_with_key`: how many individual TOC dictionaries contain that key.
- `books_with_key`: how many distinct `parent_asin` identifiers have at least one such dictionary.

**How to read the result:** `label` appears in 3,801 TOC items belonging to 204 book records. A single book can have a label on many chapters, so it contributes many items but only one book. The larger item total does **not** mean richer data; the two columns measure different things. These totals check whether a key exists, not whether its value is nonempty or useful.

In [ ]:
raw_property_rows = []
for row in raw_items.itertuples():
    if isinstance(row.raw_item, dict):
        for key, value in row.raw_item.items():
            raw_property_rows.append({
                "parent_asin": row.parent_asin,
                "raw_position": row.raw_position,
                "property": key,
                "value": value,
            })

raw_properties = pd.DataFrame(raw_property_rows)
property_summary = raw_properties.groupby("property").agg(
    items_with_key=("raw_position", "size"),
    books_with_key=("parent_asin", "nunique"),
)
display(property_summary.sort_index())

## 4. Explain count differences

Objective: Explain why the saved `toc_entries` column contains 18 fewer TOC items than the `raw_toc_json` column across our 9,034 book rows.

**What does 18 mean?** It means **18 individual TOC items spread across 17 books**. Sixteen books have one such item each; one book has two: `16 × 1 + 1 × 2 = 18`. It does not mean 18 books or 18 missing chapter titles. These items have no nonblank text in `title`, `value`, or `label` for the existing extraction to use.

For example, the `raw_toc_json` column for *Turn eBay Data into Dollars* (`parent_asin = "0072262362"`) contains 17 items. Its last item is `{"type": "/type/text", "value": ""}`. The empty string `""` contains no title to copy. The same book's `toc_entries` column contains 16 entries. We are observing a difference already present in the downloaded data; this cell does not remove anything.

**Across all books:** `raw_toc_json` contains 141,513 items; `toc_entries` contains 141,495 entries. The 18 items without main text account for the difference. Section 5 also compares the remaining text and its order; counts alone cannot prove that the right text was kept.

We also check whether the two stored count columns are accurate. `raw_toc_item_count` should equal the number of items we count in decoded `raw_toc_json`; `toc_entry_count` should equal the number we count in `toc_entries`. Both checks report **0 books with an incorrect recorded count** in this snapshot.

These checks use the columns in **data/raw/books.parquet**. The newer **data/processed/toc_features.parquet** keeps all 18 items as rows marked `missing_text = True`. They have not been deleted from our files.

**Reading 1 versus 2 in the per-book table:** `raw_items_without_main_text = 1` means that book has one TOC item with no usable title/value/label text; `2` means it has two. This is not a rating or a number of missing words. For example, `parent_asin = "0717803880"` has 6 items in `raw_toc_json` but 4 entries in `toc_entries`: its first and sixth raw items have no main text.

**Conclusion:** Sixteen book records account for one omitted item each, and one record accounts for two: 18 items across 17 records. The saved item totals are accurate; the difference comes from which items the earlier extraction retained, not a counting error. We cannot call these missing chapters because the source items supply no chapter text.

In [ ]:
candidate_text = raw_items["candidate_text"].astype("string")
raw_items["has_candidate_text"] = candidate_text.fillna("").str.strip().ne("")
# Rows in this inspection table are TOC items, not whole books.
items_without_main_text = raw_items.loc[~raw_items["has_candidate_text"]]

toc_counts = books[["parent_asin", "title", "raw_toc_shape"]].copy()
# Compare each book's saved raw count with what raw_toc_json actually contains.
toc_counts["stored_raw_count"] = pd.to_numeric(books["raw_toc_item_count"])
toc_counts["actual_raw_count"] = raw_tocs.map(len)
# Do the same for the existing toc_entries column in the raw book dataset.
toc_counts["stored_parsed_count"] = pd.to_numeric(books["toc_entry_count"])
toc_counts["actual_parsed_count"] = books["toc_entries"].map(len)
toc_counts["items_without_text"] = (
    toc_counts["parent_asin"].map(items_without_main_text.groupby("parent_asin").size()).fillna(0).astype(int)
)
toc_counts["raw_minus_parsed"] = toc_counts["actual_raw_count"] - toc_counts["actual_parsed_count"]

print(f"TOC items counted in raw_toc_json: {toc_counts['actual_raw_count'].sum():,}")
print(f"TOC entries counted in existing toc_entries: {toc_counts['actual_parsed_count'].sum():,}")
print(f"Items without title/value/label text: {len(items_without_main_text):,}")
print(f"Books containing those items: {items_without_main_text['parent_asin'].nunique():,}")
print("Books where raw_toc_item_count is incorrect:", toc_counts["stored_raw_count"].ne(toc_counts["actual_raw_count"]).sum())
print("Books where toc_entry_count is incorrect:", toc_counts["stored_parsed_count"].ne(toc_counts["actual_parsed_count"]).sum())

print("One row per affected book: compare its two TOC columns.")
book_comparison_columns = [
    "parent_asin", "title", "actual_raw_count", "actual_parsed_count", "items_without_text"
]
display(toc_counts.loc[toc_counts["raw_minus_parsed"].ne(0), book_comparison_columns].rename(columns={
    "actual_raw_count": "items_in_raw_toc_json",
    "actual_parsed_count": "entries_in_toc_entries",
    "items_without_text": "raw_items_without_main_text",
}))
print("One row per item without main text: raw_position is its position inside raw_toc_json.")
display(items_without_main_text[["parent_asin", "raw_position", "raw_item"]])

Objective: Inspect the exact raw item that accounts for one book's count difference.

Our usual example, *Never Split the Difference*, has 14 items in both columns, so it does not illustrate this issue. Here we deliberately inspect **Turn eBay Data into Dollars** (`parent_asin = "0072262362"`). Its 17th item in `raw_toc_json` contains an empty `value` string. The `toc_entries` column for the same book has 16 entries. This is one of the 17 affected books; the 18-item total covers all affected books.

In [ ]:
count_example_book = books.loc[books["parent_asin"] == "0072262362"].iloc[0]
count_example_raw_toc = json.loads(count_example_book["raw_toc_json"])
count_example_parsed_toc = count_example_book["toc_entries"]

print(f"Book: {count_example_book['title']} ({count_example_book['parent_asin']})")
print(f"Items in this book's raw_toc_json: {len(count_example_raw_toc)}")
print(f"Entries in this book's existing toc_entries: {len(count_example_parsed_toc)}")
print("Item 17 from raw_toc_json (Python index 16):")
print(json.dumps(count_example_raw_toc[16], indent=2))
print("Its value is an empty string, so it supplies no text for a parsed entry.")

## 5. Inspect parsed fields and preservation

Objective: Build `parsed_items`, an inspection table with one row for each dictionary already stored in the `books["toc_entries"]` column. Keep its book identifier and its position in that book's TOC.

Each existing dictionary has five keys: `text`, `source_key`, `sequence`, `level`, and `page`. This table lets us compare those saved values with the TOC items decoded from `raw_toc_json`. It is not a training split or a new saved dataset.

In [ ]:
parsed_rows = []
for parent_asin, entries in zip(books["parent_asin"], books["toc_entries"], strict=True):
    for parsed_position, entry in enumerate(entries, start=1):
        parsed_rows.append({
            **entry,
            "parent_asin": parent_asin,
            "parsed_position": parsed_position,
        })

parsed_items = pd.DataFrame(parsed_rows)
print(f"Parsed entries: {len(parsed_items):,}")
display(parsed_items["source_key"].value_counts(dropna=False).rename("entries").to_frame())

Objective: Check whether the existing `toc_entries` column kept the main text and selected details from the `raw_toc_json` column **for the same book, in the same order**.

For this comparison only, skip the 18 items in `raw_toc_json` without main text. Pair each remaining item with the next entry in that book's `toc_entries`. Neither saved column is changed.

| Value read from an item in `raw_toc_json` | Value compared in `toc_entries` |
| --- | --- |
| First nonblank `title`, otherwise `value`, otherwise `label`; or the item itself if it is a string | `text` |
| Name of the chosen key; `None` for a string item | `source_key` |
| `level` | `level` |
| `pagenum` | `page` |

We remove spaces at the ends of the selected main text and compare metadata numbers as text, so `0` and `"0"` agree. Missing values on both sides also agree. `preservation_review` holds all compared pairs and their results; the last displayed table shows up to 20 disagreements.

In [ ]:
usable_raw_items = raw_items.loc[raw_items["has_candidate_text"]].copy()
usable_raw_items["parsed_position"] = usable_raw_items.groupby("parent_asin").cumcount() + 1
preservation_review = usable_raw_items.merge(
    parsed_items, on=["parent_asin", "parsed_position"], how="outer", indicator=True,
    validate="one_to_one",
)
preservation_review["text_matches"] = (
    preservation_review["candidate_text"].astype("string").str.strip()
    .eq(preservation_review["text"].astype("string")).fillna(False)
)
preservation_review["source_key_matches"] = (
    preservation_review["candidate_source_key"].fillna("<missing>")
    .eq(preservation_review["source_key"].fillna("<missing>"))
)

def raw_field_as_text(item, field):
    value = item.get(field) if isinstance(item, dict) else None
    return None if value is None else str(value)


for raw_field, parsed_field in [("level", "level"), ("pagenum", "page")]:
    raw_values = preservation_review["raw_item"].map(
        lambda item: raw_field_as_text(item, raw_field)
    ).astype("string")
    parsed_values = preservation_review[parsed_field].astype("string")
    preservation_review[f"{parsed_field}_matches"] = (
        raw_values.fillna("<missing>").eq(parsed_values.fillna("<missing>"))
    )

comparison_fields = ["text_matches", "source_key_matches", "level_matches", "page_matches"]
print("Items present on both sides:", preservation_review["_merge"].eq("both").sum())
display((~preservation_review[comparison_fields]).sum().astype(int).rename("entry_disagreements").to_frame())
display(preservation_review.loc[~preservation_review[comparison_fields].all(axis=1)].head(20))

**Conclusion for this saved file:** All 141,495 remaining TOC items have a corresponding entry in `toc_entries`. There are **0 disagreements** for each of the four checks above.

This means the existing `toc_entries` agrees with the tested extraction rule for main text, its source key, level, and page, in order. It does **not** mean every original detail was kept: this check does not compare chapter subtitles or author credits. Nor does it prove that the source TOC is complete or belongs to the correct edition. The next cell examines details these four checks do not cover.

Objective: Inspect `label`, `subtitle`, `authors`, `description`, and `class` **inside TOC dictionaries decoded from the `raw_toc_json` column**. These same original TOC items are also saved under `table_of_contents` inside the `openlibrary_raw_edition_json` column.

We compare them with the separate `toc_entries` column. Its dictionaries have only `text`, `source_key`, `sequence`, `level`, and `page`—there is no separate `label` or `subtitle` key there.

For *Never Split the Difference* (`parent_asin = "0062407805"`), the first item in `raw_toc_json` has:

```python
{"label": "Chapter 1", "title": "The New Rules",
 "subtitle": "How to Become the Smartest Person in Any Room"}
# Selected keys from the first item, not its complete dictionary.
```

The corresponding `toc_entries` dictionary has `text = "The New Rules"` and `source_key = "title"`. Reading that text alone misses **"Chapter 1"** and **"How to Become the Smartest Person in Any Room"**. In other items, a label can already be the chosen `text` when both title and value are absent, so labels should not be appended blindly.

The table below measures how often each of these extra keys appears in `raw_toc_json`. The examples beneath it show actual values.

In [ ]:
additional_properties = raw_properties.loc[
    raw_properties["property"].isin(["label", "subtitle", "authors", "description", "class"])
]
display(additional_properties.groupby("property").agg(
    items_with_key=("raw_position", "size"),
    books_with_key=("parent_asin", "nunique"),
))
with pd.option_context("display.max_colwidth", 140):
    display(additional_properties.groupby("property", sort=True).head(5))

**What the extra-key table tells us**

| Key inside a TOC item in `raw_toc_json` | TOC items with the key | Book records with at least one such item | Meaning |
| --- | ---: | ---: | --- |
| `authors` | 6 | 5 | Six items contain an author-credit key, spread across five book records. |
| `class` | 69 | 5 | Sixty-nine items contain a type/category key, spread across five book records. |
| `description` | 1 | 1 | One item in one book record contains a description key. |
| `label` | 3,801 | 204 | Labels occur on many items within those 204 book records. |
| `subtitle` | 16 | 2 | Sixteen TOC items have a subtitle key, but they belong to only two book records. These are not the DataFrame's book-level subtitles. |

**Conclusion:** Reading only the `text` key of each dictionary in the `toc_entries` column misses some details retained in `raw_toc_json`. For our selected book, `selected_book["toc_entries"][0]["text"]` returns only `"The New Rules"`, without its chapter subtitle. The totals alone do not establish that every extra value is useful or that adding it improves recommendations. Preserve these details and inspect their values before deciding how to use them.

Objective: Measure missing and blank values separately for each parsed field. Percentages use all parsed entries as the denominator.

In [ ]:
coverage_rows = []
for field in ["text", "sequence", "level", "page", "source_key"]:
    values = parsed_items[field].astype("string")
    coverage_rows.append({
        "field": field,
        "missing_entries": int(values.isna().sum()),
        "blank_entries": int(values.str.strip().eq("").sum()),
        "missing_percent": round(100 * values.isna().mean(), 2),
    })
display(pd.DataFrame(coverage_rows).set_index("field"))

## 6. Inspect order, hierarchy, and pages

Objective: Check sequence numbering, observe level values, and identify books with multiple levels. A flat or missing level does not establish that chapters have no hierarchy.

In [ ]:
sequence_values = pd.to_numeric(parsed_items["sequence"], errors="coerce")
sequence_disagreements = sequence_values.ne(parsed_items["parsed_position"])
print("Sequence differs from actual position:", int(sequence_disagreements.sum()))
display(parsed_items["level"].value_counts(dropna=False).rename("entries").to_frame())

level_values = pd.to_numeric(parsed_items["level"], errors="coerce")
level_steps = level_values.groupby(parsed_items["parent_asin"]).diff()
large_level_jumps = level_steps.gt(1)
print("Entries with a downward hierarchy jump larger than one level:", int(large_level_jumps.sum()))
display(parsed_items.loc[large_level_jumps].head(10))

levels_per_book = parsed_items.groupby("parent_asin")["level"].nunique()
print("Books with multiple nonmissing levels:", int(levels_per_book.gt(1).sum()))
print("Books without any level:", int(levels_per_book.eq(0).sum()))

Objective: Inspect page values that are not simple integers, such as Roman numerals or ranges. These are review cases, not automatically invalid pages.

In [ ]:
page_values = parsed_items["page"].astype("string")
present_pages = page_values.notna() & page_values.str.strip().ne("")
integer_pages = page_values.str.fullmatch(r"[0-9]+").fillna(False)
other_pages = present_pages & ~integer_pages
print("Entries with a page:", int(present_pages.sum()))
print("Entries with a noninteger page representation:", int(other_pages.sum()))
display(page_values.loc[other_pages].value_counts().head(20).rename("entries").to_frame())

## 7. Inspect text quality and repetition

Objective: Flag text needing inspection, including blank text, markup, control characters, and numeric-only labels. Flags can overlap and do not trigger cleaning.

In [ ]:
entry_text = parsed_items["text"].astype("string")
text_flags = pd.DataFrame({
    "missing_text": entry_text.isna(),
    "blank_text": entry_text.str.strip().eq(""),
    "surrounding_whitespace": entry_text.ne(entry_text.str.strip()),
    "html_like_markup": entry_text.str.contains(r"<[^>]+>", regex=True),
    "replacement_character": entry_text.str.contains("\ufffd", regex=False),
    "control_characters": entry_text.str.contains(r"[\x00-\x08\x0b\x0c\x0e-\x1f]", regex=True),
    "numeric_only": entry_text.str.fullmatch(r"\s*[0-9]+\s*"),
    "line_breaks": entry_text.str.contains(r"[\r\n]", regex=True),
}).fillna(False)
flag_rows = []
for flag in text_flags.columns:
    flagged = text_flags[flag]
    flag_rows.append({
        "check": flag,
        "entries": int(flagged.sum()),
        "books": parsed_items.loc[flagged, "parent_asin"].nunique(),
    })
display(pd.DataFrame(flag_rows).set_index("check"))
display(parsed_items.loc[text_flags.any(axis=1), ["parent_asin", "parsed_position", "text"]].head(20))

Objective: Find repeated text within the same book. The second check ignores case and repeated whitespace only for comparison. Repeated headings can be legitimate, so no entries are removed.

In [ ]:
duplicate_review = parsed_items[["parent_asin", "parsed_position", "text"]].copy()
duplicate_review["comparison_text"] = entry_text.str.casefold().str.replace(r"\s+", " ", regex=True).str.strip()
exact_repeats = duplicate_review.duplicated(["parent_asin", "text"], keep=False)
normalized_repeats = duplicate_review.duplicated(["parent_asin", "comparison_text"], keep=False)
for name, repeated in [("Exact repeated text", exact_repeats), ("Case/whitespace-normalized repeated text", normalized_repeats)]:
    print(f"{name}: {repeated.sum():,} entries in {duplicate_review.loc[repeated, 'parent_asin'].nunique():,} books")
display(duplicate_review.loc[normalized_repeats].head(20))

Objective: Measure entry and per-book text lengths to understand future chunking needs. Whitespace word counts are descriptive and are not model token counts.

In [ ]:
text_lengths = parsed_items[["parent_asin"]].copy()
text_lengths["characters"] = entry_text.str.len()
text_lengths["whitespace_words"] = entry_text.str.split().str.len()
book_text_lengths = text_lengths.groupby("parent_asin")[["characters", "whitespace_words"]].sum()
display(text_lengths[["characters", "whitespace_words"]].describe(percentiles=[0.5, 0.9, 0.99]).T)
display(book_text_lengths.describe(percentiles=[0.5, 0.9, 0.99]).T)

fig, axes = plt.subplots(1, 2, figsize=(10, 3))
toc_counts["actual_parsed_count"].hist(bins=40, ax=axes[0])
axes[0].set(xlabel="Parsed entries per book", ylabel="Books")
text_lengths["characters"].hist(bins=40, ax=axes[1])
axes[1].set(xlabel="Characters per entry", ylabel="Entries")
plt.tight_layout()
plt.show()
display(books.loc[books["parent_asin"].isin(book_text_lengths.nlargest(5, "characters").index),
                  ["parent_asin", "title", "toc_entry_count"]])

## 8. Review what the TOC describes

Objective: Find examples mentioning volumes, parts, chapters, or common front/back matter. These English patterns are incomplete inspection aids; they do not classify the whole TOC or decide which entries to keep.

In [ ]:
content_patterns = {
    "volume_marker": r"^\s*(?:v\.|vol\.?|volume)\s+[0-9ivxlcdm]+\b",
    "part_marker": r"^\s*part\s+[0-9ivxlcdm]+\b",
    "chapter_marker": r"^\s*(?:chapter|ch\.)\s+[0-9ivxlcdm]+\b",
    "front_or_back_matter": r"^\s*(?:introduction|preface|foreword|acknowledg\w*|bibliography|index|appendix|appendices)\b",
}
for label, pattern in content_patterns.items():
    matches = entry_text.str.contains(pattern, case=False, regex=True, na=False)
    print(f"{label}: {matches.sum():,} entries; {parsed_items.loc[matches, 'parent_asin'].nunique():,} books")
    display(parsed_items.loc[matches, ["parent_asin", "parsed_position", "text"]].head(5))

Objective: Search other saved text for explicit contents headings. Matches are candidates for manual review and may just advertise a TOC; absence of a match does not rule out chapter information. Nested property names are searched separately to locate structured TOCs.

In [ ]:
def contents_key_paths(value, path=""):
    paths = set()
    if isinstance(value, dict):
        for key, child in value.items():
            child_path = f"{path}.{key}" if path else key
            if re.search(r"toc|contents?|chapters?|outline", key, re.IGNORECASE):
                paths.add(child_path)
            paths.update(contents_key_paths(child, child_path))
    elif isinstance(value, list):
        for child in value:
            paths.update(contents_key_paths(child, path + "[]"))
    return paths


key_path_rows = []
for column in ["amazon_raw_record_json", "openlibrary_raw_edition_json"]:
    for parent_asin, record in zip(books["parent_asin"], books[column].map(json.loads), strict=True):
        for path in contents_key_paths(record):
            key_path_rows.append({"source": column, "path": path, "parent_asin": parent_asin})
key_path_summary = pd.DataFrame(key_path_rows, columns=["source", "path", "parent_asin"])
display(key_path_summary.groupby(["source", "path"]).size().rename("books").to_frame())

Objective: Display possible contents references in Amazon descriptions/features and Open Library descriptions/notes. This phrase search is deliberately limited to explicit headings; inspect the full values before interpreting them as TOC content.

In [ ]:
contents_heading_pattern = r"\b(?:table\s+of\s+contents|list\s+of\s+chapters|contents\s*:)"
text_candidates = []
for position, book in enumerate(books.itertuples()):
    edition = openlibrary_records.iloc[position]
    fields = {
        "amazon_description": list(book.description),
        "amazon_features": list(book.features),
        "openlibrary_description": edition.get("description"),
        "openlibrary_notes": edition.get("notes"),
    }
    for field, value in fields.items():
        text = json.dumps(value, ensure_ascii=False)
        match = re.search(contents_heading_pattern, text, re.IGNORECASE)
        if match:
            text_candidates.append({
                "row_position": position,
                "parent_asin": book.parent_asin,
                "field": field,
                "excerpt": text[max(0, match.start() - 60):match.end() + 240],
            })
hidden_text_review = pd.DataFrame(text_candidates, columns=["row_position", "parent_asin", "field", "excerpt"])
display(hidden_text_review.groupby("field").size().rename("candidate_books").to_frame())
with pd.option_context("display.max_colwidth", None):
    display(hidden_text_review.head(10))

## 9. Confirm the raw file is unchanged

Objective: Compare the before and after checksums. Keep all cleaning, removal, imputation, feature-selection, and chunking decisions for a later phase with an agreed evaluation design.

In [ ]:
checksum_after = file_checksum(raw_path)
print(f"Checksum before: {checksum_before}")
print(f"Checksum after:  {checksum_after}")
assert checksum_after == checksum_before, "The raw Parquet changed during exploration."
print("Raw file unchanged.")

### Questions to resolve through manual inspection

- Does the TOC describe the matched edition, a multivolume work, or a subset of the book?
- Which raw labels, subtitles, author credits, or descriptions add meaning beyond parsed text?
- Do repeated headings reflect actual sections, duplicated source data, or parsing issues?
- What do absent levels/pages mean for each source format?
- Are phrase-search candidates actual chapter lists or mentions of a TOC?

No automatic correction follows from these flags. This catalog has been explored as development evidence. Define retrieval evaluation before selecting preprocessing or chunking strategies. Record findings in [docs/data.md](../docs/data.md).